In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

from robot_wm.inference.actor.base import RobotActionHistory, RobotObsHistory
from robot_wm.inference.actor.cost.visual_based_latents_cost import \
    L2VisualLatentsCost
from robot_wm.inference.robot.base import RobotObs
from robot_wm.inference.task.reference_episode import ImageProprioGoal
from robot_wm.utils.config import from_config

In [ ]:
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence

import hydra
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from IPython import display
from omegaconf import OmegaConf
from PIL import Image, ImageOps
from tqdm.notebook import tqdm, trange

# from robot_actions.utils.notebook import prepare_inputs, save_rollout, view_rollout
from robot_wm.utils.notebook import prepare_inputs, save_rollout, view_rollout

os.environ["COSMOS_HOME"] = "/fxs-cortex/carohiguera/vt_wm"

## Dataset

We start by loading the droid dataset

In [ ]:
dataset = from_config(
    "datasets/configs/datasets/droid.yaml",
    # can also use RAD_paths.csv for the heldout set.
    overrides=[
        "manifest=/fsx-cortex-datacache/shared/datasets/droid/011825/droid_h5/train_paths.csv"
    ],
)

MPK dataset without wrist camera

In [ ]:
dataset = from_config(
    "datasets/configs/datasets/mpk.yaml",
    overrides=[
        "manifest_patterns=['**/evaluation_tasks/**/folding/**/episode.h5', '**/evaluation_tasks/**/pick/**/episode.h5', '**/evaluation_tasks/**/push/**/episode.h5', '**/evaluation_tasks/**/text_goal/**/episode.h5']"
    ]
)

MPK dataset with wrist camera

In [ ]:
dataset = from_config(
    "datasets/configs/datasets/mpk.yaml",
    overrides=[
        "manifest_patterns=[**/evaluation_tasks/mpk/pick/reachliftcup_v1/run_0001/episode.h5,**/evaluation_tasks/mpk/pick/pickcube_v0/run_0001/episode.h5,**/evaluation_tasks/mpk/pick/pickpen_v0/run_0001/episode.h5,**/evaluation_tasks/mpk/push/brownboxpush_v0/run_0001/episode.h5,**/evaluation_tasks/mpk/folding/foldjacketsleeve_v1/run_0001/episode.h5]"
    ]
)

## load MPK pickingcube100 data

In [ ]:
import os
import h5py
import numpy as np

try:
    import cv2
    _HAS_CV2 = True
except Exception:
    _HAS_CV2 = False


def _h5_to_dict(node):
    if isinstance(node, h5py.Dataset):
        arr = node[()]
        if isinstance(arr, (bytes, bytearray)):
            return arr.decode("utf-8", errors="ignore")
        if isinstance(arr, np.ndarray) and arr.dtype.kind == "S":
            return arr.astype(str)
        return arr
    elif isinstance(node, h5py.Group):
        return {k: _h5_to_dict(v) for k, v in node.items()}
    else:
        return None


def _read_video_as_numpy(path, convert_to_rgb=True, max_frames=None):
    if not _HAS_CV2:
        raise ImportError("`pip install opencv-python`")
    cap = cv2.VideoCapture(path)
    frames = []
    n = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if convert_to_rgb:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
        n += 1
        if max_frames is not None and n >= max_frames:
            break
    cap.release()
    if not frames:
        return np.empty((0,), dtype=np.uint8)
    return np.stack(frames, axis=0)


def load_run_data(folder_path: str,
                  read_video: bool = True,
                  video_to_rgb: bool = True,
                  max_video_frames: int | None = None):
    data = {}

    # episode.h5
    h5_file = os.path.join(folder_path, "episode.h5")
    if os.path.exists(h5_file):
        with h5py.File(h5_file, "r") as f:
            data["episode_h5"] = _h5_to_dict(f)

    # episode.mp4
    mp4_file = os.path.join(folder_path, "episode.mp4")
    if os.path.exists(mp4_file):
        if read_video:
            data["episode_mp4"] = _read_video_as_numpy(
                mp4_file, convert_to_rgb=video_to_rgb, max_frames=max_video_frames
            )
        else:
            data["episode_mp4"] = mp4_file

    # success.txt
    txt_file = os.path.join(folder_path, "success.txt")
    if os.path.exists(txt_file):
        with open(txt_file, "r", encoding="utf-8") as f:
            data["success"] = f.read().strip()

    # trajectory.hdf5
    traj_file = os.path.join(folder_path, "trajectory.hdf5")
    if os.path.exists(traj_file):
        with h5py.File(traj_file, "r") as f:
            data["trajectory"] = _h5_to_dict(f)
    return data


## gripper state
- for MPK-pickingcube100 dataset, open means 0, close means 1. works for WM.
- Krishna's data, gripper state is raw value from sensor, 0-0.085. need normalization.

In [ ]:


def sample_to_obs_act_history(sample, resize_and_crop=(180, 320)):
    
    obs = sample["episode_data"]["observation"]
    
    observation_history = RobotObsHistory(max_context=1000, freq=30)
    for i in range(len(obs["joint_position"])):
        joints = obs["joint_position"][i]
        ee_pose = obs["cartesian_position"][i]
        gripper = obs["gripper_position"][i]
        ee_pose_with_gripper = np.concatenate((ee_pose, np.array([gripper])))
        if i == 20:
            print("ee_pose_with_gripper:", ee_pose_with_gripper)

        cam_1 = obs["exterior_image_1_left"][i]
        cam_2 = obs["exterior_image_2_left"][i]
        cam_3 = obs["wrist_image_left"][i]
        cam_3 = cv2.resize(cam_3, (640, 480))  
        cam = np.concatenate([cam_1, cam_2, cam_3], axis=0)
        cam = np.transpose(cam, (2, 0, 1)) / 255.0
        if resize_and_crop is not None:
            import torchvision
            import torch
            transform = torchvision.transforms.Compose([
                torchvision.transforms.Resize(resize_and_crop),
                torchvision.transforms.CenterCrop(resize_and_crop),
            ])
            cam = transform(torch.tensor(cam)).float().cpu().numpy()
        robot_obs = RobotObs(joints, ee_pose_with_gripper, cam)
        observation_history.push(robot_obs)
    
    action_history = observation_history.get_action_history_from_ee_deltas()
    return observation_history, action_history

# len(dataset)

In [ ]:
# mpk pickingcube100
path = "/home/hanchencui/projects/robot_world_models/MPK_data/MPK-pickcube100/data/evaluation_tasks/mpk/pick/pickcube100/run_0035"
run_data = load_run_data(path, read_video=True)
sample = run_data["episode_h5"]

In [ ]:
MODEL_CROP = [182*3, 322]
# sample = dataset[1]
observation_history, action_history = sample_to_obs_act_history(sample, resize_and_crop=MODEL_CROP)
# observation_history, action_history = episode_to_obs_act_history(sample, resize_and_crop=MODEL_CROP)
observation_history.show()

## World Model
Instantiate a world model from our model zoo or _menagerie_. 

> Each world model requires you to have setup their corresponding project, see corresponding README's or Cortex Wiki for more details.

In [ ]:
# can add overrides to load different weights or change params
wm_config = 'menagerie/config/DINOv2-ST-1B-DROID-rollout-3v.yaml'
wm = from_config(wm_config)
# wm = from_config('menagerie/config/dino_wm.yaml')
# wm = from_config("menagerie/config/jepa_wm.yaml")

#### Does the world model encode correctly an observation stream?

We encode observations from droid, no rollouts. This should look very similar to the dataset video.

In [ ]:
# Some models don't allow a very large context for rollouts, you could see and change that parameter for this visualization at wm.max_context
wm.max_context = 1000
ans = wm.encode_history(observation_history, action_history)
imgs = wm.decode_latents(ans)
imgs.show()

#### Does the WM does sensible future predictions?
Now encode a small history and make predictions predictions using GT actions.

In [ ]:
init, context, end = 0, 20, -1

# Take 20 frames of context
context_obs = observation_history[init : init + context]
context_actions = context_obs.get_action_history_from_ee_deltas()

# Encode them
history = wm.encode_history(context_obs, context_actions)

# Take all the future GT actions
future_obs = observation_history[init + context :]  # can be used for comparison
future_actions = future_obs.get_action_history_from_ee_deltas()

# Rollout and show them
rollout = wm.rollout(history, future_actions)
pred_imgs = wm.decode_latents(rollout)
print(pred_imgs.obs[0].image.shape)   
pred_imgs.show()
# print(pred_imgs.obs[0].image.shape)
# future_obs.show()
# imgs.show()

## one step inference

### action promitive inference

In [ ]:
# repeat zero action 5 times
one_step_action = []
dummy_action = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.05]
for _ in range(5):
    one_step_action.append(dummy_action)

one_step_action = torch.tensor(one_step_action).reshape(1, 1, 35).float()


new_context = history
for i in range(5):
    # one_step_action = future_action_sequences[:, i*5:(i+1)*5, :].reshape(1,1,35)
    new_context, z_new_visual = wm._one_step_latent_pred(new_context, one_step_action)
rollout = wm.decode_latents(new_context)
rollout.show()

### load preprocess functions

In [ ]:
from PIL import Image
def resize_with_pad(images: np.ndarray, height: int, width: int, method=Image.BILINEAR) -> np.ndarray:
    """Replicates tf.image.resize_with_pad for multiple images using PIL. Resizes a batch of images to a target height.

    Args:
        images: A batch of images in [..., height, width, channel] format.
        height: The target height of the image.
        width: The target width of the image.
        method: The interpolation method to use. Default is bilinear.

    Returns:
        The resized images in [..., height, width, channel].
    """
    # If the images are already the correct size, return them as is.
    if images.shape[-3:-1] == (height, width):
        return images

    original_shape = images.shape

    images = images.reshape(-1, *original_shape[-3:])
    resized = np.stack([_resize_with_pad_pil(Image.fromarray(im), height, width, method=method) for im in images])
    return resized.reshape(*original_shape[:-3], *resized.shape[-3:])


def _resize_with_pad_pil(image: Image.Image, height: int, width: int, method: int) -> Image.Image:
    """Replicates tf.image.resize_with_pad for one image using PIL. Resizes an image to a target height and
    width without distortion by padding with zeros.

    Unlike the jax version, note that PIL uses [width, height, channel] ordering instead of [batch, h, w, c].
    """
    cur_width, cur_height = image.size
    if cur_width == width and cur_height == height:
        return image  # No need to resize if the image is already the correct size.

    ratio = max(cur_width / width, cur_height / height)
    resized_height = int(cur_height / ratio)
    resized_width = int(cur_width / ratio)
    resized_image = image.resize((resized_width, resized_height), resample=method)

    zero_image = Image.new(resized_image.mode, (width, height), 0)
    pad_height = max(0, int((height - resized_height) / 2))
    pad_width = max(0, int((width - resized_width) / 2))
    zero_image.paste(resized_image, (pad_width, pad_height))
    assert zero_image.size == (width, height)
    return zero_image

## WM inference server
- pip install Pyro5 (Pyro5 is a python version RPC)
- change the host with the current machine, ```daemon = Pyro5.api.Daemo(host="a100-st-p4de24xlarge-132")  ```
- copy the output URI to client

In [ ]:
import Pyro5.api
import numpy as np
from matplotlib import pyplot as plt
import cv2
import time
# from scipy.spatial.transform import Rotation as R



INFERENCE_STEPS = 3
@Pyro5.api.expose
class ControllerInterface:
    def __init__(self):
        self.step_count = 0
        self.new_context = history

    def step(self, data_dict):  # data_dict: {'action': [...], 'step': int}
        action_chunk = data_dict["action"]
        


        # rollout 3 WM steps
        for i in range(INFERENCE_STEPS):
            one_action_chunk = torch.tensor(action_chunk[i*5:(i+1)*5])  # Ensure action_chunk is a tensor
            one_step_action = one_action_chunk.reshape(1, 1, 35)
            self.new_context, z_new_visual = wm._one_step_latent_pred(self.new_context, one_step_action)
            self.step_count += 1
        rollout = wm.decode_latents(self.new_context)
        last_image = rollout.obs[-1].image
        last_image = (last_image * 255).astype(np.uint8)

        # process rollout image
        img = np.transpose(last_image, (1, 2, 0))
        h = img.shape[0] // 3
        img_right = img[0*h:1*h, :, :]
        img_left = img[1*h:2*h, :, :]
        img_wrist = img[2*h:3*h, :, :]
        img_left = resize_with_pad(img_left.astype(np.uint8), 256, 256)
        img_wrist = resize_with_pad(img_wrist.astype(np.uint8), 256, 256)
        img_right = resize_with_pad(img_right.astype(np.uint8), 256, 256)


        # print(f"control Step {self.step_count} | Received action: {action}")



        # output gif every 30 steps
        if self.step_count % 30 == 0:
            rollout.show()
        return {
            "left_image": img_left.tolist(),  # Convert to list for serialization
            "right_image": img_right.tolist(),
            "wrist_image": img_wrist.tolist(),
            "step": self.step_count,
        }

# Pyro5 server
daemon = Pyro5.api.Daemon(host="a100-st-p4de24xlarge-483")
uri = daemon.register(ControllerInterface)
print("Controller server running at:")
print(uri)
daemon.requestLoop()